<a href="https://colab.research.google.com/github/usama488/bioinformatics-analysis/blob/main/SingleCell_RNASeq_Immune_Microenvironment_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Single-Cell RNA-Seq Analysis of Immune Cell Populations (Tumor Microenvironment Methodology)

**Project 12 of Advanced Bioinformatics

> **Honesty note:** `pbmc3k` real blood immune cells hain (tumor biopsy nahi) — lekin isi exact pipeline (QC → clustering → UMAP → marker-based annotation) ko **real tumor scRNA-seq datasets** (jaise GEO accessions `GSE72056` melanoma, `GSE131907` lung cancer) par bhi apply kiya jata hai. Yeh dataset is liye chuna gaya kyunke yeh reliably publicly downloadable hai aur wahi immune cell types capture karta hai jo TME mein milte hain.

---

## 📋 Table of Contents

| Section | Content |
|---|---|
| 1 | Setup & Installation |
| 2 | Load Real Single-Cell Data (10x Genomics) |
| 3 | Quality Control Metrics |
| 4 | Filtering Low-Quality Cells |
| 5 | Normalization & Log Transformation |
| 6 | Highly Variable Gene Selection |
| 7 | Dimensionality Reduction (PCA) |
| 8 | Clustering (Leiden Algorithm) |
| 9 | UMAP Embedding & Visualization |
| 10 | Marker Gene Identification |
| 11 | Cell Type Annotation |
| 12 | Immune Cell Composition Analysis |
| 13 |  **Runtime Prediction** — Apna Cell Expression Profile Daal Kar Cell Type Predict Karein |


## 1. Setup & Installation




In [1]:
# Pinned, compatible versions — avoids numba/numpy/pandas conflicts in Colab
!pip install -q "numpy>=1.24,<2.5" "scipy>=1.11,<1.13" scanpy leidenalg python-igraph plotly scikit-learn ipywidgets

print("=" * 70)
print(" Installation complete (pinned versions to avoid conflicts).")
print("  IMPORTANT: Ab Colab runtime RESTART karein:")
print("    Menu: Runtime → Restart session (ya Ctrl+M .)")
print("    Restart ke baad is cell ko dobara run KAREIN NAHI —")
print("    seedha NEECHE wali cell (imports) se continue karein.")
print("=" * 70)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 76.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take i

> 🔁 **Runtime Restart Zaroori Hai**

>
> **Upar wali cell run karne ke baad:**
> 1. Colab menu se **Runtime → Restart session** click karein (ya keyboard shortcut `Ctrl+M .`)
> 2. Restart hone ke baad **upar wali install cell dobara run na karein**
> 3. Seedha **neeche wali imports cell** se aage continue karein
>
> Agar phir bhi error aaye, neeche wali "Fallback Fix" cell run karein.


In [1]:
# Agar upar wali install ke baad bhi koi conflict/ImportError aaye, yeh cell use karein
# (exact pinned combo — numba ke sath compatible), phir dobara Runtime -> Restart session karein.

# !pip install -q --force-reinstall "numpy==1.26.4" "scipy==1.12.0" "pandas==2.1.4" scanpy leidenalg python-igraph


In [2]:
import numpy as np
import pandas as pd
import scanpy as sc
import warnings
warnings.filterwarnings("ignore")

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

sc.settings.verbosity = 1
np.random.seed(42)

print(" Sab imports successful — scanpy version:", sc.__version__)


✅ Sab imports successful — scanpy version: 1.11.5


## 2. Load Real Single-Cell Data (10x Genomics PBMC3k)

Yeh **real sequencing data** hai — 2,700 human peripheral blood mononuclear cells, 10x Genomics Chromium platform se, publicly released aur `scanpy` ke through directly downloadable.


In [4]:
try:
    adata = sc.datasets.pbmc3k()
    data_source = " Real 10x Genomics PBMC3k dataset loaded successfully"
except Exception as e:
    print(f" Could not download real dataset ({e}) — generating a structurally-realistic fallback.")
    n_cells, n_genes = 2700, 1000
    rng = np.random.default_rng(42)
    import anndata
    X = rng.poisson(1.5, size=(n_cells, n_genes)).astype(float)
    gene_names = [f"GENE{i}" for i in range(n_genes)]
    for marker in ["CD3D", "CD3E", "CD8A", "IL7R", "CD79A", "MS4A1", "NKG7", "GNLY", "LYZ", "CD14", "FCGR3A", "MS4A7", "FCER1A", "PPBP"]:
        gene_names[rng.integers(0, n_genes)] = marker
    adata = anndata.AnnData(X=X, var=pd.DataFrame(index=gene_names))
    adata.var_names_make_unique()
    data_source = " Simulated fallback dataset"

adata.var_names_make_unique()
print(data_source)
print(f"Dataset shape: {adata.shape[0]} cells x {adata.shape[1]} genes")


 Real 10x Genomics PBMC3k dataset loaded successfully
Dataset shape: 2700 cells x 32738 genes


## 3. Quality Control Metrics

In [5]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

qc_df = adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].copy()

fig = make_subplots(rows=1, cols=3, subplot_titles=("Genes per Cell", "Total Counts per Cell", "% Mitochondrial Reads"))
fig.add_trace(go.Violin(y=qc_df["n_genes_by_counts"], box_visible=True, meanline_visible=True,
                         fillcolor="#2E86AB", opacity=0.6, name=""), row=1, col=1)
fig.add_trace(go.Violin(y=qc_df["total_counts"], box_visible=True, meanline_visible=True,
                         fillcolor="#43AA8B", opacity=0.6, name=""), row=1, col=2)
fig.add_trace(go.Violin(y=qc_df["pct_counts_mt"], box_visible=True, meanline_visible=True,
                         fillcolor="#E63946", opacity=0.6, name=""), row=1, col=3)
fig.update_layout(height=450, title_text="Quality Control Metrics (Before Filtering)", showlegend=False)
fig.show()

fig2 = px.scatter(qc_df, x="total_counts", y="n_genes_by_counts", color="pct_counts_mt",
                   title="QC: Total Counts vs Genes Detected (colored by %MT)",
                   template="plotly_white", color_continuous_scale="Reds")
fig2.update_layout(height=450)
fig2.show()


## 4. Filtering Low-Quality Cells

In [6]:
n_cells_before = adata.n_obs

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[adata.obs["pct_counts_mt"] < 20, :].copy()
adata = adata[adata.obs["n_genes_by_counts"] < 4000, :].copy()

n_cells_after = adata.n_obs
print(f"Cells before filtering: {n_cells_before}")
print(f"Cells after filtering:  {n_cells_after}  ({n_cells_before - n_cells_after} removed as low-quality)")
print(f"Genes remaining: {adata.n_vars}")


Cells before filtering: 2700
Cells after filtering:  2698  (2 removed as low-quality)
Genes remaining: 13714


## 5. Normalization & Log Transformation

In [7]:
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

print("Normalization complete: total-count normalized (CP10K) + log1p transformed")


Normalization complete: total-count normalized (CP10K) + log1p transformed


## 6. Highly Variable Gene Selection

In [8]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

hvg_df = adata.var[["means", "dispersions_norm", "highly_variable"]].copy()
fig = px.scatter(hvg_df, x="means", y="dispersions_norm", color="highly_variable",
                  title=f"Highly Variable Gene Selection ({hvg_df['highly_variable'].sum()} genes selected)",
                  template="plotly_white", color_discrete_map={True: "#E63946", False: "#cccccc"},
                  labels={"means": "Mean Expression", "dispersions_norm": "Normalized Dispersion"})
fig.update_layout(height=450)
fig.show()

adata.raw = adata
adata = adata[:, adata.var.highly_variable].copy()
print(f"Retained {adata.n_vars} highly variable genes for downstream analysis")


Retained 1865 highly variable genes for downstream analysis


## 7. Scaling & Dimensionality Reduction (PCA)

In [9]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver="arpack", n_comps=30)

var_ratio = adata.uns["pca"]["variance_ratio"][:15]
fig = px.bar(x=[f"PC{i+1}" for i in range(15)], y=var_ratio,
             title="PCA — Variance Explained by Top 15 Principal Components",
             labels={"x": "Principal Component", "y": "Variance Ratio"}, template="plotly_white",
             color=var_ratio, color_continuous_scale="Viridis")
fig.update_layout(height=400, showlegend=False)
fig.show()


## 8. Clustering (Leiden Algorithm)

In [10]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=30)
sc.tl.leiden(adata, resolution=0.5)

n_clusters = adata.obs["leiden"].nunique()
cluster_counts = adata.obs["leiden"].value_counts().sort_index()
print(f"Number of clusters identified: {n_clusters}")

fig = px.bar(cluster_counts, title="Cell Count per Cluster", template="plotly_white",
             labels={"value": "Cell Count", "index": "Cluster"},
             color=cluster_counts.values, color_continuous_scale="Tealgrn")
fig.update_layout(height=400, showlegend=False)
fig.show()


Number of clusters identified: 8


## 9. UMAP Embedding & Visualization

In [11]:
sc.tl.umap(adata)

umap_df = pd.DataFrame(adata.obsm["X_umap"], columns=["UMAP1", "UMAP2"])
umap_df["cluster"] = adata.obs["leiden"].values
umap_df["n_genes"] = adata.obs["n_genes_by_counts"].values
umap_df["total_counts"] = adata.obs["total_counts"].values

fig = px.scatter(umap_df, x="UMAP1", y="UMAP2", color="cluster",
                  title="UMAP Embedding — Colored by Leiden Cluster",
                  template="plotly_white", hover_data=["n_genes", "total_counts"],
                  color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_traces(marker=dict(size=5, opacity=0.8))
fig.update_layout(height=600)
fig.show()


## 10. Marker Gene Identification

In [12]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")

top_markers = {}
for cluster in adata.obs["leiden"].cat.categories:
    names = adata.uns["rank_genes_groups"]["names"][cluster][:5]
    top_markers[cluster] = list(names)

marker_summary = pd.DataFrame(top_markers).T
marker_summary.columns = [f"Top Marker {i+1}" for i in range(5)]
marker_summary.index.name = "Cluster"
marker_summary


,Top Marker 1,Top Marker 2,Top Marker 3,Top Marker 4,Top Marker 5
Cluster,,,,,
0,RPS12,LDHB,RPS25,RPS27,RPS6
1,S100A9,S100A8,LYZ,TYROBP,FTL
2,NKG7,CST7,GZMA,B2M,CTSW
3,CD74,CD79A,HLA-DRA,CD79B,HLA-DPB1
4,LST1,COTL1,FCER1G,AIF1,FTH1
5,HLA-DPA1,HLA-DPB1,HLA-DRA,HLA-DRB1,CD74
6,PPBP,GPX1,CALM3,MYL6,SDPR
7,TYMS,KIAA0101,ZWINT,GAPDH,SNRNP25


## 11. Cell Type Annotation (Canonical Marker Genes)

In [13]:
canonical_markers = {
    "CD4+ T Cell": ["IL7R", "CD3D", "CD3E"],
    "CD8+ T Cell": ["CD8A", "CD3D", "CD3E"],
    "B Cell": ["CD79A", "MS4A1", "CD79B"],
    "NK Cell": ["NKG7", "GNLY", "KLRD1"],
    "CD14+ Monocyte": ["LYZ", "CD14", "S100A8"],
    "FCGR3A+ Monocyte": ["FCGR3A", "MS4A7"],
    "Dendritic Cell": ["FCER1A", "CST3"],
    "Platelet": ["PPBP", "PF4"],
}

available_genes = set(adata.raw.var_names)

def score_cluster_celltype(cluster_markers, marker_dict, available_genes):
    scores = {}
    for cell_type, markers in marker_dict.items():
        overlap = len(set(markers) & set(cluster_markers) & available_genes)
        scores[cell_type] = overlap
    if max(scores.values()) == 0:
        return "Unidentified"
    return max(scores, key=scores.get)

cluster_annotations = {}
for cluster, markers in top_markers.items():
    cluster_annotations[cluster] = score_cluster_celltype(markers, canonical_markers, available_genes)

adata.obs["cell_type"] = adata.obs["leiden"].map(cluster_annotations)
umap_df["cell_type"] = adata.obs["cell_type"].values

print("Cluster -> Cell Type mapping:")
for c, ct in cluster_annotations.items():
    print(f"  Cluster {c}: {ct}  (top markers: {', '.join(top_markers[c][:3])})")


Cluster -> Cell Type mapping:
  Cluster 0: Unidentified  (top markers: RPS12, LDHB, RPS25)
  Cluster 1: CD14+ Monocyte  (top markers: S100A9, S100A8, LYZ)
  Cluster 2: NK Cell  (top markers: NKG7, CST7, GZMA)
  Cluster 3: B Cell  (top markers: CD74, CD79A, HLA-DRA)
  Cluster 4: Unidentified  (top markers: LST1, COTL1, FCER1G)
  Cluster 5: Unidentified  (top markers: HLA-DPA1, HLA-DPB1, HLA-DRA)
  Cluster 6: Platelet  (top markers: PPBP, GPX1, CALM3)
  Cluster 7: Unidentified  (top markers: TYMS, KIAA0101, ZWINT)


In [16]:
fig = px.scatter(umap_df, x="UMAP1", y="UMAP2", color="cell_type",
                  title="UMAP — Annotated Immune Cell Types",
                  template="plotly_white", color_discrete_sequence=px.colors.qualitative.Bold)
fig.update_traces(marker=dict(size=6, opacity=0.85, line=dict(width=0.3, color='white')))
fig.update_layout(height=600)
fig.show()


## 12. Immune Cell Composition Analysis

In [17]:
composition = adata.obs["cell_type"].value_counts()

fig = px.pie(names=composition.index, values=composition.values, hole=0.45,
             title="Immune Cell Type Composition", template="plotly_white",
             color_discrete_sequence=px.colors.qualitative.Bold)
fig.update_layout(height=500)
fig.show()

# Marker gene expression dot-plot style visualization
key_markers = ["IL7R", "CD8A", "CD79A", "MS4A1", "NKG7", "GNLY", "LYZ", "CD14", "FCGR3A", "FCER1A", "PPBP"]
key_markers = [m for m in key_markers if m in adata.raw.var_names]

expr_matrix = pd.DataFrame(
    adata.raw[:, key_markers].X.toarray() if hasattr(adata.raw[:, key_markers].X, "toarray") else adata.raw[:, key_markers].X,
    columns=key_markers, index=adata.obs_names
)
expr_matrix["cell_type"] = adata.obs["cell_type"].values
mean_expr_by_type = expr_matrix.groupby("cell_type")[key_markers].mean()

fig2 = px.imshow(mean_expr_by_type.values, x=key_markers, y=mean_expr_by_type.index,
                  color_continuous_scale="Reds", aspect="auto",
                  title="Mean Marker Gene Expression by Annotated Cell Type",
                  labels=dict(color="Mean Expression (log-norm)"))
fig2.update_layout(height=450, xaxis_tickangle=-45)
fig2.show()


## 13.  Runtime Prediction — Apna Cell Expression Profile Daal Kar Cell Type Predict Karein

Ek **classifier train** kar rahe hain jo key marker genes ki expression se cell type predict karta hai — phir aap kisi bhi naye cell ka expression profile daal kar uska type turant predict kar saktay hain.


In [18]:
# Train a classifier on the annotated single-cell data using key marker gene expression
clf_data = expr_matrix.copy()
le_cell = LabelEncoder()
X_cells = clf_data[key_markers]
y_cells = le_cell.fit_transform(clf_data["cell_type"])

X_train, X_test, y_train, y_test = train_test_split(X_cells, y_cells, test_size=0.25, stratify=y_cells, random_state=42)

cell_scaler = StandardScaler()
X_train_s = cell_scaler.fit_transform(X_train)
X_test_s = cell_scaler.transform(X_test)

cell_clf = RandomForestClassifier(n_estimators=300, random_state=42)
cell_clf.fit(X_train_s, y_train)

preds = cell_clf.predict(X_test_s)
print(f"Cell-type classifier accuracy: {accuracy_score(y_test, preds):.3f}")
print(classification_report(y_test, preds, target_names=le_cell.classes_, zero_division=0))


Cell-type classifier accuracy: 0.919
                precision    recall  f1-score   support

        B Cell       0.93      0.95      0.94        87
CD14+ Monocyte       0.87      0.98      0.93       113
       NK Cell       0.88      0.88      0.88       112
      Platelet       1.00      0.50      0.67         4
  Unidentified       0.94      0.91      0.93       359

      accuracy                           0.92       675
     macro avg       0.93      0.84      0.87       675
  weighted avg       0.92      0.92      0.92       675



In [19]:
marker_boxes = {}
marker_widgets = []
marker_defaults = clf_data[key_markers].mean().to_dict()
for gene in key_markers:
    box = widgets.FloatSlider(value=round(marker_defaults[gene], 2), min=0, max=8, step=0.1,
                               description=gene, style={'description_width': '80px'},
                               layout=widgets.Layout(width='340px'))
    marker_boxes[gene] = box
    marker_widgets.append(box)

predict_btn = widgets.Button(description=" Cell Type Predict Karein", button_style='success',
                              layout=widgets.Layout(width='260px', height='42px'))
out = widgets.Output()

def render_cell_result(label, proba, classes):
    top3_idx = proba.argsort()[::-1][:3]
    color = "#2E86AB"
    rows_html = "".join(
        f'<div style="display:flex; justify-content:space-between; font-size:14px; margin-top:5px;">'
        f'<span>{classes[i]}</span><span><b>{proba[i]*100:.1f}%</b></span></div>'
        for i in top3_idx
    )
    html = f"""
    <div style="border:2px solid {color}; border-radius:12px; padding:18px; margin-top:12px; font-family:sans-serif; background:#fafafa;">
        <div style="font-size:21px; font-weight:700; color:{color};"> Predicted Cell Type: {classes[top3_idx[0]]}</div>
        <div style="font-size:13px; margin-top:10px; color:#333;"><b>Top 3 Probabilities:</b></div>
        {rows_html}
    </div>
    """
    display(HTML(html))

def on_predict(b):
    with out:
        clear_output()
        row = {gene: marker_boxes[gene].value for gene in key_markers}
        row_df = pd.DataFrame([row])[key_markers]
        row_scaled = cell_scaler.transform(row_df)
        proba = cell_clf.predict_proba(row_scaled)[0]
        render_cell_result(cell_clf.predict(row_scaled)[0], proba, le_cell.classes_)

predict_btn.on_click(on_predict)

display(widgets.HTML("<b style='font-size:15px;'>Marker Gene Expression (log-normalized)</b>"))
display(widgets.GridBox(marker_widgets, layout=widgets.Layout(grid_template_columns="repeat(2, 350px)", grid_gap="6px")))
display(predict_btn)
display(out)


HTML(value="<b style='font-size:15px;'>Marker Gene Expression (log-normalized)</b>")

GridBox(children=(FloatSlider(value=0.87, description='IL7R', layout=Layout(width='340px'), max=8.0, style=Sli…

Button(button_style='success', description=' Cell Type Predict Karein', layout=Layout(height='42px', width='26…

Output()